In [ ]:
"""
Data Cleaning Pipeline — Mental Health Text Classification
==========================================================
Thứ tự xử lý:
  1. ftfy      : sửa mojibake / encoding lỗi
  2. emoji     : chuyển emoji → text mô tả (tiếng Anh) (mỗi mô hình lưu ý xử lý kỹ chỗ này cho embedding :tag:)
  3. Các bước  : URL -> <url>, @mention -> <user> , #hashtag -> hashtag, repeated chars, whitespace

Ghi chú thiết kế:
- Giữ nguyên CASE (BERT cần; lowercase tuỳ model sau)
- Giữ dấu câu cơ bản (!, ?, ...) vì EDA cho thấy chúng có tín hiệu phân lớp
- Không xoá stopword ở đây (để cho TF-IDF / tokenizer quyết định)
"""

'\nData Cleaning Pipeline — Mental Health Text Classification\n==========================================================\nThứ tự xử lý:\n  1. ftfy      : sửa mojibake / encoding lỗi\n  2. emoji     : chuyển emoji → text mô tả (tiếng Anh) (mỗi mô hình lưu ý xử lý kỹ chỗ này cho embedding)\n  3. Các bước  : URL -> <url>, @mention -> <user> , #hashtag -> HASH_TOKEN hashtag (có dùng thêm wordsegment để tách ví dụ như INeedHelp -> I Need Help), repeated chars, whitespace\n\nGhi chú thiết kế:\n- Giữ nguyên CASE (BERT cần; lowercase tuỳ model sau)\n- Giữ dấu câu cơ bản (!, ?, ...) vì EDA cho thấy chúng có tín hiệu phân lớp\n- Không xoá stopword ở đây (để cho TF-IDF / tokenizer quyết định)\n'

In [2]:
!pip install ftfy emoji wordsegment

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 2.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 608.4/608.4 kB 10.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.8/4.8 MB 69.9 MB/s eta 0:00:00


In [3]:
import re
import ftfy
import emoji
import html
import pandas as pd
from typing import Optional
import unicodedata

from wordsegment import load, segment
load()

In [4]:
# ── 1. Sửa mojibake & encoding lỗi ──────────────────────────────────────────

def fix_encoding(text: str) -> str:
    """
    Dùng ftfy để sửa các lỗi encoding phổ biến:
      - mojibake (VD: "cafÃ©" → "café")
      - ký tự bị escape sai
      - byte sequences lỗi từ latin-1 / windows-1252
    """
    return ftfy.fix_text(text)

In [5]:
# ── 2. Chuyển emoji → text mô tả ────────────────────────────────────────────

def convert_emoji(text: str, language: str = "en") -> str:
    """
    Chuyển emoji thành mô tả văn bản.
    VD: "I'm so sad 😢" → "I'm so sad :crying_face:"

    Dùng dấu space để tránh dính vào từ bên cạnh, sau đó
    chuẩn hóa lại khoảng trắng thừa.
    """
    # demojize thêm dấu : : xung quanh tên emoji
    text = emoji.demojize(text, language=language)
    # Xoá dấu : để lấy text thuần (VD: :crying_face: → crying face)
    # HOẶC giữ nguyên :tag: nếu muốn downstream model nhận dạng
    # → ở đây giữ nguyên :tag: vì BERT / TF-IDF có thể học từ chúng
    text = text.replace("::", ": :")
    return text

In [6]:
# ── 3. Xoá URL ───────────────────────────────────────────────────────────────

def clean_urls(text: str) -> str:
    """
    Xoá http/https/www links. Thay bằng " <url> " để giữ word boundary.
    VD: "check this https://t.co/xxx now" → "check this  <url>  now"
    """
    pattern = r"https?://\S+|www\.\S+"
    return re.sub(pattern, " <url> ", text)

In [7]:
# ── 4. Xoá @mention và #hashtag ─────────────────────────────────────────────

def clean_mentions(text: str) -> str:
    """
    @username  → <user> (social media artifact)
    #hashtag   → giữ lại phần text, bỏ dấu #
                 VD: #MentalHealth → MentalHealth (vẫn mang nghĩa)
    """
    text = re.sub(r"@\w+", " <user> ", text)             # xoá @mention
    return text

In [8]:
# ── 5. Clean hashtag ─────────────────────────────────────────────
# Regex phân tách CamelCase trước khi segment
def split_camel(text: str) -> str:
    # VD: "SavePalestine" -> "Save Palestine"
    text = re.sub(r'([a-z])([A-Z])', r'\1 \2', text)
    text = re.sub(r'([A-Z]+)([A-Z][a-z])', r'\1 \2', text)
    return text

def is_noise_hashtag(body: str) -> bool:
    """Drop hashtag không có giá trị ngữ nghĩa"""
    # Chỉ số hoặc ký tự lạ
    if re.fullmatch(r'[\d\W_]+', body):
        return True
    # Quá ngắn
    if len(body) <= 2:
        return True
    # Random alphanumeric (bot/spam)
    if re.fullmatch(r'[a-zA-Z0-9]{3,6}', body) and re.search(r'\d', body) and re.search(r'[a-zA-Z]', body):
        # VD: 3bD5a, x200B, eJ0Po
        return True
    return False

def clean_hashtags(text: str) -> str:
    hashtags = re.findall(r'#(\w+)', text)

    for body in hashtags:
        if is_noise_hashtag(body):
            # Xóa hẳn hashtag rác
            text = re.sub(r'#' + re.escape(body) + r'\b', '', text)
            continue

        # Bước 1: Tách CamelCase trước
        camel_split = split_camel(body)

        # Bước 2: wordsegment cho phần còn lại
        words = segment(camel_split.lower())
        body_clean = ' '.join(words)

        text = re.sub(r'#' + re.escape(body) + r'\b', body_clean, text)

    return text.strip()

In [9]:
# ── 6. Chuẩn hoá repeated characters ────────────────────────────────────────

def normalize_repeated_chars(text: str, max_repeat: int = 2) -> str:
    """
    Giảm ký tự lặp liên tiếp xuống còn max_repeat.
    VD: "sooooo tired" → "soo tired"  (giữ 1 ký tự emphasis)
        "ahhhhhhh"     → "ahh"
    Giữ max_repeat=2 thay vì 1 để bảo toàn tín hiệu cảm xúc.
    """
    pattern = rf"([a-zA-Z])\1{{{max_repeat},}}"
    replacement = r"\1" * max_repeat
    return re.sub(pattern, replacement, text)

In [10]:
# ── 7. Chuẩn hoá Unicode còn lại ────────────────────────────────────────────

def normalize_unicode(text: str) -> str:
    """
    Sau khi ftfy sửa mojibake và emoji đã được chuyển đổi,
    xử lý các ký tự Unicode còn sót:
      - Dấu nháy fancy → dấu nháy thẳng
      - Dấu gạch ngang em/en → dấu gạch thường
      - Xoá ký tự điều khiển (control characters)
    Không xoá toàn bộ non-ASCII vì một số ký tự hợp lệ (VD: accented letters
    sau khi ftfy xử lý đúng) vẫn mang nghĩa.
    """
    # Dấu nháy fancy
    text = re.sub(r"[\u2018\u2019\u201a\u201b]", "'", text)   # ' ' ‚ ‛
    text = re.sub(r"[\u201c\u201d\u201e\u201f]", '"', text)   # " " „ ‟
    # Dấu gạch ngang
    text = re.sub(r"[\u2013\u2014\u2015]", "-", text)         # – — ―
    # Dấu chấm lửng
    text = re.sub(r"\u2026", "...", text)                      # …
    # Control characters (trừ \n, \t)
    text = re.sub(r"[\x00-\x08\x0b\x0c\x0e-\x1f\x7f]", " ", text)
    return text

In [11]:
# ── 8. Chuẩn hoá whitespace ──────────────────────────────────────────────────

def normalize_whitespace(text: str) -> str:
    """
    - Thay \n, \t, \r bằng space
    - Thu gọn nhiều space liên tiếp thành 1
    - Strip đầu/cuối
    """
    text = re.sub(r"[\n\t\r]", " ", text)
    text = re.sub(r" {2,}", " ", text)
    return text.strip()

In [12]:
# ── 9. Chuẩn hoá whitespace ──────────────────────────────────────────────────

def normalize_to_ascii(text: str) -> str:
    """
    Chuyển đổi các ký tự Unicode có dấu về dạng ASCII tương ứng.
    Ví dụ: 'résumé' -> 'resume'
    """
    # Phân tách các ký tự (NFD: Normalization Form Decomposition)
    # Ví dụ: 'é' sẽ tách thành 'e' và dấu sắc '´'
    normalized = unicodedata.normalize('NFD', text)

    # Mã hóa sang ASCII và bỏ qua các ký tự không thể chuyển đổi (như các dấu vừa tách ra)
    ascii_text = normalized.encode('ascii', 'ignore').decode('utf-8')

    return ascii_text

In [13]:
# ── Pipeline tổng hợp ────────────────────────────────────────────────────────

def clean_text(
    text: str,
    emoji_language: str = "en",
    keep_case: bool = True,
) -> str:
    """
    Pipeline đầy đủ theo đúng thứ tự:
      ftfy → emoji → url → mention/hashtag
      → repeated chars → unicode → whitespace → (lowercase)

    Args:
        text          : văn bản gốc
        emoji_language: ngôn ngữ mô tả emoji ('en' mặc định)
        keep_case     : True để giữ case (cho BERT),
                        False để lowercase (cho TF-IDF / LSTM truyền thống)
    """
    if not isinstance(text, str) or not text.strip():
        return ""

    text = fix_encoding(text)
    text = convert_emoji(text, emoji_language)
    text = clean_urls(text)
    text = clean_mentions(text)
    text = clean_hashtags(text)
    text = normalize_repeated_chars(text)
    text = normalize_unicode(text)
    text = normalize_to_ascii(text)
    text = normalize_whitespace(text)

    if not keep_case:
        text = text.lower()

    return text

In [14]:
# ── Áp dụng lên DataFrame ────────────────────────────────────────────────────

def clean_dataframe(
    df: pd.DataFrame,
    text_col: str = "text",
    emoji_language: str = "en",
    keep_case: bool = True,
) -> pd.DataFrame:
    """
    Áp dụng clean_text lên toàn bộ DataFrame.
    Thêm cột:
      - text_clean  : văn bản đã làm sạch
      - is_short    : flag văn bản rất ngắn (≤ 3 từ)
      - word_count  : số từ sau khi clean
    """
    df = df.copy()
    df["text_clean"] = df[text_col].apply(
        lambda t: clean_text(t, emoji_language=emoji_language, keep_case=keep_case)
    )

    return df

In [15]:
df_train = pd.read_csv("/content/train_split.csv", encoding='utf-8')
df_val = pd.read_csv("/content/val_split.csv", encoding='utf-8')
df_test = pd.read_csv("/content/test_split.csv", encoding='utf-8')

In [16]:
df_train['text'] = clean_dataframe(df_train)['text_clean']
df_val['text'] = clean_dataframe(df_val)['text_clean']
df_test['text'] = clean_dataframe(df_test)['text_clean']

In [18]:
# ── 6. Lưu lại thành file CSV
df_train.to_csv('train_clean.csv', index=False, encoding='utf-8')
df_val.to_csv('val_clean.csv',   index=False, encoding='utf-8')
df_test.to_csv('test_clean.csv',  index=False, encoding='utf-8')
print('\n💾 Đã lưu: train_clean.csv | val_clean.csv | test_clean.csv')


💾 Đã lưu: train_clean.csv | val_clean.csv | test_clean.csv
